# Conversational RAG with Chroma

This notebook builds one end-to-end retrieval-augmented generation workflow:

1. Turn sample text into LangChain `Document` objects.
2. Split the documents, embed them, and store them in Chroma.
3. Retrieve relevant chunks for a question.
4. Use separate message prompts to rewrite follow-up questions and answer from retrieved context.


## 1. Imports and model setup

Set `OPENAI_API_KEY` in the project `.env` file. This example uses OpenAI for generation and a local Hugging Face model for embeddings.

In [185]:
import os

from dotenv import load_dotenv
from langchain_classic.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in the project .env file before running this cell.")

llm = ChatOpenAI(model="gpt-4o-mini")


## 2. Prepare source documents

Keep each source as a separate document so retrieval can find the relevant facts. Metadata records where a chunk came from.

In [186]:
sample_data = [
    "Artificial intelligence (AI) is the field of building computer systems that perform tasks such as recognizing patterns, understanding language, and making predictions. Machine learning is a branch of AI where models learn patterns from examples. Deep learning uses multi-layer neural networks to learn complex patterns from data.",
    "A good foundation for learning AI includes Python programming, basic statistics, probability, and linear algebra. Practice with NumPy and pandas, and build intuition by experimenting with small datasets and simple models.",
    "Supervised machine learning uses labeled examples. Regression predicts numeric values, while classification predicts categories. Unsupervised learning finds structure in unlabeled data, such as grouping examples with clustering. Use separate training and test data to evaluate a model on examples it has not seen.",
    "Neural networks learn by adjusting weights to reduce errors. Transformers use attention to relate parts of a sequence. Large language models generate text from learned patterns, but their answers can be inaccurate and important claims should be checked against reliable sources.",
    "Retrieval-augmented generation (RAG) combines document search with a language model. Documents are split into chunks and converted into vector embeddings. For each question, relevant chunks are retrieved and provided to the model as context. RAG can improve answers but does not guarantee correctness.",
    "Responsible AI practice includes protecting personal information, checking for bias, testing realistic cases, and explaining limitations. Use data only with permission, and keep qualified people involved in high-impact decisions.",
    "Ajay was born on July 7, 1991. He enjoys playing cricket, cooking, and driving cars and bikes.",
    "Ajay is a software developer learning AI. He is married, has one child, and lives in a family of five.",
]

documents = [
    Document(page_content=text, metadata={"source": f"sample-{index}"})
    for index, text in enumerate(sample_data, start=1)
]
print(f"Prepared {len(documents)} source documents.")


Prepared 8 source documents.


## 3. Split, embed, and index

Overlapping chunks keep nearby context together. The same embedding model must be used when indexing and querying.

In [187]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
)
chunks = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_learning_demo",
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9942.90it/s]


Indexed 8 chunks.


## 4. Prompts and conversational retrieval

The two prompts have different jobs:

- **Question-rewrite prompt:** uses prior messages to turn a follow-up into a standalone search query. It needs `chat_history` and `input`.
- **Answer prompt:** answers the current question using retrieved documents. It needs `context` (required by `create_stuff_documents_chain`) and `input`; history is optional.

Keep these prompts separate: the retriever needs a search query, while the answer chain needs retrieved context.

In [188]:
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given the conversation history and latest question, rewrite the question as a standalone search query. Do not answer it.",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    rewrite_prompt,
)

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer using only the retrieved context. If the context does not contain the answer, say you do not know.\n\nRetrieved context:\n{context}",
        ),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
    ]
)
answer_chain = create_stuff_documents_chain(llm, answer_prompt)
conversational_rag = create_retrieval_chain(
    history_aware_retriever,
    answer_chain,
)


## 5. Ask questions and keep chat history

Pass the current question as a string. Store each user question and assistant answer as messages so follow-up questions can be rewritten with context.

In [189]:
chat_history = []


def ask(question: str) -> dict:
    result = conversational_rag.invoke(
        {"input": question, "chat_history": chat_history}
    )
    chat_history.extend(
        [
            HumanMessage(content=question),
            AIMessage(content=result["answer"]),
        ]
    )
    return result


first_result = ask("Who is Ajay and what does he enjoy?")
print(first_result["answer"])
print("Sources:", sorted({doc.metadata["source"] for doc in first_result["context"]}))

follow_up_result = ask("When was he born?")
print(follow_up_result["answer"])


Ajay is a software developer learning AI. He enjoys playing cricket, cooking, and driving cars and bikes.
Sources: ['sample-5', 'sample-7', 'sample-8']
Ajay was born on July 7, 1991.
